# [0.6] How to Know When an Interpretability Result Is Fake - Solutions

Reference validation notebook for the fake-result diagnostics section.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter0_fundamentals"
section = "part6_fake_interpretability_results"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_fake_interpretability_results.tests as tests
from chapter0_fundamentals.exercises.part6_fake_interpretability_results import solutions

In [ ]:
tests.test_binary_accuracy_thresholds_signed_scores(solutions.binary_accuracy)
tests.test_label_leakage_report_flags_direct_label_feature(solutions.label_leakage_report)
tests.test_cherry_pick_report_compares_selected_to_population(solutions.cherry_pick_report)
tests.test_probe_overfit_report_requires_heldout_gap(solutions.probe_overfit_report)
tests.test_random_direction_control_report_rejects_weak_claim(solutions.random_direction_control_report)
tests.test_fake_result_audit_report_aggregates_all_failure_modes(solutions.fake_result_audit_report)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["contract_passed"]
assert contract["leakage"]["leaked_feature_accuracy"] == 1.0
assert contract["leakage"]["shifted_no_leak_accuracy"] == 0.0
assert contract["cherry_pick"]["inflation_ratio"] >= 3.0
assert contract["probe_overfit"]["train_accuracy"] == 1.0
assert contract["probe_overfit"]["heldout_accuracy"] == 0.5
assert contract["random_direction"]["detects_random_direction_failure"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["leaked_feature_accuracy"] == 1.0
assert gpu["shifted_no_leak_accuracy"] == 0.0
assert gpu["leakage_gap"] >= 0.5
assert gpu["cherry_pick_inflation"] >= 3.0
assert gpu["selected_mean_effect"] >= 5 * gpu["population_median_effect"]
assert gpu["probe_train_accuracy"] == 1.0
assert gpu["probe_heldout_accuracy"] == 0.5
assert gpu["probe_overfit_gap"] >= 0.35
assert gpu["random_direction_control_rejects_claim"]
assert gpu["random_direction_effect_gap"] < 0.25
assert gpu["all_bogus_results_flagged"]
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "device",
    "leaked_feature_accuracy",
    "shifted_no_leak_accuracy",
    "cherry_pick_inflation",
    "probe_overfit_gap",
    "random_direction_effect_gap",
    "peak_vram_gb",
]}